In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_name = 'mps'
    pipeline_device = 'mps'
elif torch.cuda.is_available():
    device = torch.device('cuda')
    device_name = 'cuda'
    pipeline_device = 0
else:
    device = torch.device('cpu')
    device_name = 'cpu'
    pipeline_device = -1

print({'selected_device': device_name})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])


In [ ]:
model_name = 'valhalla/distilbart-mnli-12-1'

classifier = pipeline(
    task='zero-shot-classification',
    model=model_name,
    device=pipeline_device
)

print({'model_name': model_name, 'device': device_name, 'num_labels': len(class_names)})


In [ ]:
texts = dataset['text']
true_ids = np.array(dataset['label'])

batch_size = 32
all_pred_labels = []
all_pred_scores = []

for start in range(0, len(texts), batch_size):
    batch_texts = texts[start:start + batch_size]
    outputs = classifier(
        batch_texts,
        candidate_labels=class_names,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
        max_length=128
    )
    if isinstance(outputs, dict):
        outputs = [outputs]
    for out in outputs:
        all_pred_labels.append(out['labels'][0])
        all_pred_scores.append(float(out['scores'][0]))

label_to_id = {label: i for i, label in enumerate(class_names)}
pred_ids = np.array([label_to_id[label] for label in all_pred_labels])

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': all_pred_labels,
    'predicted_score': all_pred_scores
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device_name,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
